In [1]:
import numpy as np

In [2]:
def f(x): # Target function
    return x ** 2 / (x ** 3 + 1.0)

In [ ]:
def dev(f, x0, h, scheme): # Finite difference derivative
    match scheme:
        case 'forward':
            return (f(x0 + h) - f(x0)) / h
        case 'backward':
            return (f(x0) - f(x0 - h)) / h
        case 'central':
            return (f(x0 + h) - f(x0 - h)) / (2.0 * h)
        case _:
            return 0.0

In [4]:
def dev_adaptive(f, x0, h0 = 0.1, eps = 1e-06, scheme = 'forward'): # Adaptive finite difference

    h = 2.0 * h0
    j = 0
    err2 = eps + 1.0

    while err2 > eps: # Absolute error criterion

        h /= 2.0
        j += 1

        derivative1 = dev(f, x0, h, scheme)
        derivative2 = dev(f, x0, h / 2.0, scheme)
        diff21 = derivative2 - derivative1
        err2 = np.abs(diff21) if (scheme == 'forward' or scheme == 'backward') else np.abs(diff21) / 3.0 # Richardson extrapolation applied for derivative2

    rel_err2 = err2 / np.abs(derivative2) if derivative2 != 0.0 else np.nan # Relative error
    
    # Quality factor of the error. Better when is close to zero
    derivative4 = dev(f, x0, h / 4.0, scheme)
    diff42 = derivative4 - derivative2
    ratio = diff21 / diff42 if diff42 != 0.0 else np.inf
    diffQ = np.abs(2.0 - np.abs(ratio)) if (scheme == 'forward' or scheme == 'backward') else np.abs(4.0 - np.abs(ratio))

    print(f"Loop ended at j = {j}. h = {h}\n")
    return derivative2, err2, rel_err2, diffQ # It returns the derivative, the absolute error and relative error committed

In [5]:
x0 = 1.0 # Point target

In [6]:
dev_adaptive( # Forward scheme
    f = f,
    x0 = x0,
    scheme = 'forward'
)

Loop ended at j = 16. h = 3.0517578125e-06



(0.24999904635478742,
 np.float64(9.536961442790926e-07),
 np.float64(3.8147991289760756e-06),
 np.float64(0.0005341880341882543))

$$
    f'(1)_{forward} = 0.249999 \pm 0.000001
$$

In [7]:
dev_adaptive( # Backward scheme
    f = f,
    x0 = x0,
    scheme = 'backward'
)

Loop ended at j = 16. h = 3.0517578125e-06



(0.2500009536743164,
 np.float64(9.53677954385057e-07),
 np.float64(3.814697265625e-06),
 np.float64(0.0001907523271782452))

$$
    f'(1)_{backward} = 0.250001 \pm 0.000001
$$

In [8]:
dev_adaptive( # Central scheme
    f = f,
    x0 = x0,
    scheme = 'central'
)

Loop ended at j = 6. h = 0.003125



(0.2500007629354428,
 np.float64(7.629194313333679e-07),
 np.float64(3.0516684124295055e-06),
 np.float64(7.855757793917562e-05))

$$
    f'(1)_{central} = 0.2500008 \pm 0.0000008
$$

In [9]:
dev_adaptive( # Central scheme with smaller epsilon
    f = f,
    x0 = x0,
    eps = 1e-12,
    scheme = 'central'
)

Loop ended at j = 20. h = 1.9073486328125e-07



(0.24999999994179234, np.float64(0.0), np.float64(0.0), np.float64(4.0))

The quality factor is 0, that means the Richardson error estimation has failed. $\epsilon$ is too small. Let's implement a different adaptive version without $\epsilon$

In [10]:
def dev_adaptive_Q(f, x0, h0 = 0.1, scheme = 'forward', max_iter = 100):
    
    # Initialization
    h = h0

    derivative1 = dev(f, x0, h, scheme)
    derivative2 = dev(f, x0, h / 2.0, scheme)
    derivative4 = dev(f, x0, h / 4.0, scheme)

    diff21 = derivative2 - derivative1
    diff42 = derivative4 - derivative2

    if diff42 == 0.0:
        diffQ_old = np.inf
    else:
        ratio = diff21 / diff42
        diffQ_old = (
            np.abs(2.0 - np.abs(ratio))
            if scheme in ('forward', 'backward')
            else np.abs(4.0 - np.abs(ratio))
        )

    # Save best values
    h_best = h
    derivative2_best = derivative2
    diff21_best = diff21

    for j in range(max_iter):
        
        h /= 2.0

        derivative1 = dev(f, x0, h, scheme)
        derivative2 = dev(f, x0, h / 2.0, scheme)
        derivative4 = dev(f, x0, h / 4.0, scheme)

        diff21 = derivative2 - derivative1
        diff42 = derivative4 - derivative2

        if diff42 == 0.0:
            diffQ_new = np.inf
            diffQ_old = diffQ_new
        else:
            ratio = diff21 / diff42
            diffQ_new = (
                np.abs(2.0 - np.abs(ratio))
                if scheme in ('forward', 'backward')
                else np.abs(4.0 - np.abs(ratio))
            )

            if diffQ_new < diffQ_old:
                diffQ_old = diffQ_new
                h_best = h
                derivative2_best = derivative2
                diff21_best = diff21
            else:
                break

    err2 = (
        np.abs(diff21_best)
        if scheme in ('forward', 'backward')
        else np.abs(diff21_best) / 3.0
    )
    rel_err2 = (
        err2 / np.abs(derivative2_best)
        if derivative2_best != 0.0
        else np.nan
    )

    dictionary = {
        'number of iterations': j + 1,
        'h:': h_best,
        'derivative:': derivative2_best,
        'absolute error:': err2,
        'relative error': rel_err2,
        'diffQ': diffQ_old,
        'status': 'converged' if j <= max_iter else 'maximum number of iterations reached'
    }

    return dictionary

In [11]:
dev_adaptive_Q(
    f = f,
    x0 = x0,
    scheme = 'forward'
)

{'number of iterations': 13,
 'h:': 2.44140625e-05,
 'derivative:': 0.2499923706636764,
 'absolute error:': np.float64(7.6292644735076465e-06),
 'relative error': np.float64(3.051798922204536e-05),
 'diffQ': np.float64(3.5763332570937223e-06),
 'status': 'converged'}

In [12]:
dev_adaptive_Q(
    f = f,
    x0 = x0,
    scheme = 'backward'
)

{'number of iterations': 14,
 'h:': 1.220703125e-05,
 'derivative:': 0.2500038147172745,
 'absolute error:': np.float64(3.814720912487246e-06),
 'relative error': np.float64(1.5258650820193507e-05),
 'diffQ': np.float64(4.7683306552137594e-06),
 'status': 'converged'}

In [13]:
dev_adaptive_Q(
    f = f,
    x0 = x0,
    scheme = 'central'
)

{'number of iterations': 8,
 'h:': 0.00078125,
 'derivative:': 0.2500000476835851,
 'absolute error:': np.float64(4.768368218795634e-08),
 'relative error': np.float64(1.9073469237216964e-07),
 'diffQ': np.float64(1.688806589950076e-05),
 'status': 'converged'}